# HealthConnect Clinic — Week 8: Final ML Pipeline Integration & Showcase

**Intern:** Grai Rudolf  
**Role:** Junior Machine Learning Engineer  
**Track:** Machine Learning Engineering  
**Project:** HealthConnect Clinic Experience Lab — Reducing Patient No-Shows  
**Date:** September 2026  
**HC-POD Collaborators:** Mairame Samba Niang (Data Science), Data Analytics, Generative AI, Project Management  

---

## Executive Summary

This notebook documents the **Week 8 Final Integration, Presentation, and Project Showcase** for the Machine Learning Engineering track. Throughout Weeks 4–7, we transitioned from ML problem framing and leakage detection to modular pipeline architecture, comprehensive error-handling, and reliability testing. 

In Week 8, we achieve final end-to-end integration:
1. **Zero Patient Contamination:** Implementation of patient-level grouped train/test splitting (`GroupShuffleSplit`) and 5-fold `GroupKFold` cross-validation on `patient_id`.
2. **Data Science Handoff Compatibility:** Direct alignment with Mairame Samba Niang's retrained Random Forest specification (ROC-AUC ≈ 0.682, balanced weights, zero post-arrival leakage).
3. **Cost-Sensitive High-Recall Thresholding:** Operating threshold tuned to **0.35**, capturing **94.2%** of all missed appointments to optimize provider schedules.
4. **Actionable Clinical Decision Engine:** Automated mapping of risk probabilities into High, Moderate, and Low risk tiers mapped to multi-channel interventions (WhatsApp + Buffer Slot, SMS, and Email).
5. **Ethical AI & Subgroup Fairness Audit:** Quantitative evaluation of demographic parity across gender groups to address known fairness limitations.
6. **Exhaustive QA & Contract Verification:** 21 automated pytest tests verifying schema contracts, reliability shields, and artifact round-trips.

---
## 1. Setup & Environment Configuration

In [1]:
import sys
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Add week 8 root to sys.path
notebook_dir = os.getcwd()
if notebook_dir not in sys.path:
    sys.path.insert(0, notebook_dir)

from pipeline.config import (
    DATA_PATH,
    DEFAULT_THRESHOLD,
    RECOMMENDED_THRESHOLD_RANGE,
    PREDICTION_INPUT_COLS,
    RISK_TIERS,
    logger,
)
from pipeline.data_processing import (
    load_data,
    clean_data,
    patient_level_split,
    build_preprocessing_pipeline,
    get_feature_names,
)
from pipeline.feature_engineering import engineer_features
from pipeline.model_training import get_candidate_models, train_models_with_group_cv
from pipeline.evaluation import evaluate_model, audit_gender_fairness
from pipeline.validation import (
    validate_raw_data,
    validate_clean_data,
    validate_features,
    validate_prediction_input,
    validate_prediction_output,
    ValidationError,
)
from pipeline.model_integration import HealthConnectPipeline

print("Environment loaded successfully. Pipeline ready for execution.")

Environment loaded successfully. Pipeline ready for execution.


---
## 2. Data Ingestion & Post-Arrival Leakage Prevention

In Week 4 and confirmed during cross-track reviews in Week 7, we identified that `waiting_time_minutes` represents a catastrophic data leakage feature. Because waiting time occurs *after* patient check-in at the clinic, it is physically unavailable at the time of appointment booking. 

Below, we load the raw dataset, validate its schema, and demonstrate active leakage stripping during data cleaning.

In [2]:
raw_df = load_data()
validate_raw_data(raw_df)
print(f"Raw dataset shape: {raw_df.shape}")
print("Contains 'waiting_time_minutes' in raw data?", 'waiting_time_minutes' in raw_df.columns)

cleaned_df = clean_data(raw_df)
validate_clean_data(cleaned_df)
print(f"Cleaned dataset shape: {cleaned_df.shape}")
print("Contains 'waiting_time_minutes' after cleaning?", 'waiting_time_minutes' in cleaned_df.columns)
print(f"Baseline no-show prevalence: {cleaned_df['target'].mean():.4f}")

2026-09-27 15:52:15 [healthconnect.pipeline] INFO: Loaded raw data: 5000 rows, 18 columns from /home/rudolf/Desktop/projects/analyst_lab_internship/week 8/../week 4/HealthConnect_Appointment_Data.csv


2026-09-27 15:52:15 [healthconnect.pipeline] INFO: Raw data validation passed: 5000 rows, 18 columns.


2026-09-27 15:52:15 [healthconnect.pipeline] INFO: Dropped post-arrival leakage feature 'waiting_time_minutes'.


2026-09-27 15:52:15 [healthconnect.pipeline] INFO: Data cleaned: 4737 rows remaining, target mean: 0.512


2026-09-27 15:52:15 [healthconnect.pipeline] INFO: Cleaned data validation passed: 4737 rows, target balance: 0.512.


Raw dataset shape: (5000, 18)
Contains 'waiting_time_minutes' in raw data? True
Cleaned dataset shape: (4737, 15)
Contains 'waiting_time_minutes' after cleaning? False
Baseline no-show prevalence: 0.5115


---
## 3. Patient-Level Grouped Train/Test Split

Standard random train/test splitting introduces patient contamination when individuals hold multiple appointments over time. To ensure realistic evaluation on truly *unseen* patients, we enforce a patient-level split (`GroupShuffleSplit` on `patient_id`).

In [3]:
featured_df = engineer_features(cleaned_df)
train_df, test_df = patient_level_split(featured_df, group_col="patient_id", test_size=0.2, random_state=42)

overlap = set(train_df["patient_id"]).intersection(set(test_df["patient_id"]))
print(f"Train appointments: {len(train_df)} | Unique patients: {train_df['patient_id'].nunique()}")
print(f"Test appointments : {len(test_df)} | Unique patients: {test_df['patient_id'].nunique()}")
print(f"Patient overlap between train and test: {len(overlap)} (Zero leakage confirmed!)")

2026-09-27 15:52:15 [healthconnect.pipeline] INFO: Feature engineering complete: 4 domain features constructed.


2026-09-27 15:52:15 [healthconnect.pipeline] INFO: Patient-level grouped split complete: Train=3771 appointments (1344 unique patients), Test=966 appointments (336 unique patients).


Train appointments: 3771 | Unique patients: 1344
Test appointments : 966 | Unique patients: 336
Patient overlap between train and test: 0 (Zero leakage confirmed!)


---
## 4. Preprocessing Pipeline Construction

Our preprocessing pipeline combines median imputation with `RobustScaler` for numerical metrics, and most-frequent imputation with `OneHotEncoder(handle_unknown='ignore')` for categorical features to guarantee runtime stability on unobserved categories.

In [4]:
preprocessor = build_preprocessing_pipeline()
X_train = train_df.drop(columns=["target", "patient_id", "appointment_id"])
y_train = train_df["target"].values
groups_train = train_df["patient_id"].values

X_test = test_df.drop(columns=["target", "patient_id", "appointment_id"])
y_test = test_df["target"].values

X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc = preprocessor.transform(X_test)
feature_names = get_feature_names(preprocessor)

print(f"Transformed training matrix shape: {X_train_proc.shape}")
print(f"Transformed test matrix shape    : {X_test_proc.shape}")
print(f"Total features after encoding   : {len(feature_names)}")

Transformed training matrix shape: (3771, 33)
Transformed test matrix shape    : (966, 33)
Total features after encoding   : 33


---
## 5. Model Training & 5-Fold GroupKFold Cross-Validation

We benchmark candidate models using 5-fold `GroupKFold` cross-validation:
- **Logistic Regression (Baseline)**: Linear baseline model.
- **Random Forest (Final DS Spec)**: Tuned ensemble aligned with Data Science handoff (150 trees, max_depth=8, balanced class weights).
- **Gradient Boosting**: Sequential ensemble comparator.

In [5]:
candidate_models = get_candidate_models()
cv_results = train_models_with_group_cv(
    candidate_models,
    X_train_proc,
    y_train,
    groups=groups_train,
    n_splits=5,
    scoring="roc_auc"
)

summary_rows = []
for name, res in cv_results.items():
    summary_rows.append({
        "Model": name,
        "GroupKFold Mean ROC-AUC": f"{res['cv_mean']:.4f}",
        "Std Dev": f"{res['cv_std']:.4f}",
    })
pd.DataFrame(summary_rows)

2026-09-27 15:52:15 [healthconnect.pipeline] INFO: Evaluating Logistic Regression (Baseline) with 5-fold GroupKFold CV (scoring=roc_auc)...


2026-09-27 15:52:17 [healthconnect.pipeline] INFO: [Logistic Regression (Baseline)] Group CV roc_auc: 0.6735 (+/- 0.0183)


2026-09-27 15:52:17 [healthconnect.pipeline] INFO: Evaluating Random Forest (Final DS Spec) with 5-fold GroupKFold CV (scoring=roc_auc)...


2026-09-27 15:52:20 [healthconnect.pipeline] INFO: [Random Forest (Final DS Spec)] Group CV roc_auc: 0.6697 (+/- 0.0156)


2026-09-27 15:52:20 [healthconnect.pipeline] INFO: Evaluating Gradient Boosting with 5-fold GroupKFold CV (scoring=roc_auc)...


2026-09-27 15:52:23 [healthconnect.pipeline] INFO: [Gradient Boosting] Group CV roc_auc: 0.6595 (+/- 0.0118)


,Model,GroupKFold Mean ROC-AUC,Std Dev
0,Logistic Regression (Baseline),0.6735,0.0183
1,Random Forest (Final DS Spec),0.6697,0.0156
2,Gradient Boosting,0.6595,0.0118


---
## 6. Out-of-Sample Evaluation & Cost-Sensitive Threshold Tuning

HealthConnect operations prioritize high sensitivity (Recall) on missed appointments. Capturing a no-show allows the clinic to dispatch proactive WhatsApp confirmations and schedule buffer slots. Therefore, we evaluate at the Data Science recommended threshold of **0.35**.

In [6]:
rf_model = cv_results["Random Forest (Final DS Spec)"]["model"]
y_prob_rf = rf_model.predict_proba(X_test_proc)[:, 1]
y_pred_rf = (y_prob_rf >= 0.35).astype(int)

eval_metrics = evaluate_model(y_test, y_pred_rf, y_prob_rf, model_name="Random Forest (Final DS Spec)")
print(json.dumps(eval_metrics, indent=2))

2026-09-27 15:52:23 [healthconnect.pipeline] INFO: [Random Forest (Final DS Spec)] Evaluation: ROC-AUC=0.6846 | Recall=0.9420 | Precision=0.5385 | F1=0.6852 | Brier=0.2256


{
  "model_name": "Random Forest (Final DS Spec)",
  "accuracy": 0.5673,
  "precision": 0.5385,
  "recall": 0.942,
  "specificity": 0.1925,
  "f1": 0.6852,
  "roc_auc": 0.6846,
  "pr_auc": 0.6664,
  "brier_score": 0.2256,
  "confusion_matrix": {
    "tn": 93,
    "fp": 390,
    "fn": 28,
    "tp": 455
  }
}


---
## 7. Actionable Clinical Decision Engine

We stratify predictions into operational risk tiers mapped to specific clinical workflows:
- **HIGH_RISK (P >= 0.70):** Trigger WhatsApp confirmation bot + reserve contingency buffer slot.
- **MODERATE_RISK (0.35 <= P < 0.70):** Dispatch automated 48-hour SMS confirmation reminder.
- **LOW_RISK (P < 0.35):** Standard clinic email notification.

In [7]:
pipeline = HealthConnectPipeline(threshold=0.35)
pipeline.preprocessor = preprocessor
pipeline.model = rf_model
pipeline.model_name = "Random Forest (Final DS Spec)"
pipeline.feature_names = feature_names
pipeline.evaluation_metrics = eval_metrics
pipeline._fitted = True

test_predictions = pipeline.predict_batch(test_df)
print("Sample Risk-Tier Predictions:")
print(test_predictions.head(10).to_string())

print("\nOperational Tier Distribution:")
print(test_predictions["risk_tier"].value_counts(normalize=True).apply(lambda x: f"{x*100:.1f}%"))

2026-09-27 15:52:23 [healthconnect.pipeline] INFO: Feature engineering complete: 4 domain features constructed.


2026-09-27 15:52:23 [healthconnect.pipeline] INFO: Prediction output schema successfully validated.


Sample Risk-Tier Predictions:
  appointment_id  no_show_probability      risk_tier                            recommended_action
0       HC-00010               0.5049  MODERATE_RISK  TRIGGER_SMS_REMINDER_REQUESTING_CONFIRMATION
1       HC-00017               0.5215  MODERATE_RISK  TRIGGER_SMS_REMINDER_REQUESTING_CONFIRMATION
2       HC-00020               0.5129  MODERATE_RISK  TRIGGER_SMS_REMINDER_REQUESTING_CONFIRMATION
3       HC-00022               0.4481  MODERATE_RISK  TRIGGER_SMS_REMINDER_REQUESTING_CONFIRMATION
4       HC-00023               0.6165  MODERATE_RISK  TRIGGER_SMS_REMINDER_REQUESTING_CONFIRMATION
5       HC-00026               0.3372       LOW_RISK                   STANDARD_EMAIL_NOTIFICATION
6       HC-00034               0.3229       LOW_RISK                   STANDARD_EMAIL_NOTIFICATION
7       HC-00035               0.2110       LOW_RISK                   STANDARD_EMAIL_NOTIFICATION
8       HC-00041               0.4225  MODERATE_RISK  TRIGGER_SMS_REMINDER_REQU

---
## 8. Ethical AI & Subgroup Fairness Audit (Gender Disparity)

Data Science flagged a known gender fairness disparity during Week 7 testing. Here, we quantitatively audit the model across gender subgroups on the unseen test set.

In [8]:
fairness_df = audit_gender_fairness(test_df, y_test, y_pred_rf, y_prob_rf)
fairness_df

2026-09-27 15:52:23 [healthconnect.pipeline] INFO: Fairness audit completed for gender subgroups:
              gender  sample_size  prevalence  recall  precision     fpr  roc_auc
0             Female          461      0.5011  0.9524     0.5392  0.8174   0.6966
1               Male          479      0.5052  0.9339     0.5433  0.8017   0.6719
2  Prefer not to say           26      0.3846  0.9000     0.4286  0.7500   0.7125


,gender,sample_size,prevalence,recall,precision,fpr,roc_auc
0,Female,461,0.5011,0.9524,0.5392,0.8174,0.6966
1,Male,479,0.5052,0.9339,0.5433,0.8017,0.6719
2,Prefer not to say,26,0.3846,0.9000,0.4286,0.7500,0.7125


---
## 9. Live Microservice & Single-Record Inference Demonstration

Below, we simulate real-time API calls across low, moderate, and high-risk patients, and verify error-handling boundaries.

In [9]:
sample_patient = {
    "appointment_id": "HC-LIVE-001",
    "gender": "Female",
    "age": 42,
    "age_group": "35-44",
    "appointment_type": "Specialist Consultation",
    "appointment_day": "Tuesday",
    "appointment_time": "Afternoon",
    "booking_lead_days": 18,
    "previous_appointments": 4,
    "previous_no_shows": 2,
    "reminder_sent": "Yes",
    "reminder_channel": "SMS",
    "distance_to_clinic_km": 14.2,
}

prediction_result = pipeline.predict_single(sample_patient)
print("Microservice JSON Response:")
print(json.dumps(prediction_result, indent=2))

2026-09-27 15:52:23 [healthconnect.pipeline] INFO: Feature engineering complete: 4 domain features constructed.


Microservice JSON Response:
{
  "appointment_id": "HC-LIVE-001",
  "no_show_probability": 0.4962,
  "risk_tier": "MODERATE_RISK",
  "recommended_action": "TRIGGER_SMS_REMINDER_REQUESTING_CONFIRMATION"
}


---
## 10. Pipeline Artifact Serialization & Round-Trip Persistence

In [10]:
artifacts_dir = os.path.join(notebook_dir, "artifacts")
pipeline.save_artifacts(artifacts_dir)

loaded_pipeline = HealthConnectPipeline.load_artifacts(artifacts_dir)
loaded_result = loaded_pipeline.predict_single(sample_patient)

assert loaded_result["no_show_probability"] == prediction_result["no_show_probability"]
print("Artifact round-trip successfully verified! Serialized pipeline ready for deployment.")

2026-09-27 15:52:23 [healthconnect.pipeline] INFO: Pipeline artifacts successfully exported to: /home/rudolf/Desktop/projects/analyst_lab_internship/week 8/artifacts


2026-09-27 15:52:23 [healthconnect.pipeline] INFO: Loaded pipeline artifact successfully from: /home/rudolf/Desktop/projects/analyst_lab_internship/week 8/artifacts (Model: Random Forest (Final DS Spec))


2026-09-27 15:52:23 [healthconnect.pipeline] INFO: Feature engineering complete: 4 domain features constructed.


Artifact round-trip successfully verified! Serialized pipeline ready for deployment.
